#**Introdução a Redes Neurais com TensorFlow e Keras**

Importação das bibliotecas, classes e métodos utilizados ao longo da aula:

In [ ]:
import os
import random
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay)
import matplotlib.pyplot as plt


Definição de uma semente aleatória fixa:

In [ ]:
RANDOM_STATE = 7895


In [ ]:
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
tf.config.experimental.enable_op_determinism()


##**Carregando os dados**

Leitura do dataset a partir de um arquivo CSV armazenado em um repositório GitHub:

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vqrca/ml-datasets/refs/heads/main/BankChurners_portugues.csv')


Exibição das primeiras observações para verificar a estrutura dos dados e as variáveis disponíveis:

In [ ]:
df.head()


### **Preparando os dados para a rede neural**

Redes neurais **não interpretam valores categóricos**. Se codificássemos `Escolaridade` como 0, 1, 2, 3, a rede assumiria que existe uma **ordem e uma distância matemática** entre as categorias — o que raramente é verdade. Por isso, usamos o **`OneHotEncoder`**: cada categoria vira uma coluna binária independente (0 ou 1), sem impor qualquer relação artificial entre elas.

O alvo `Target` também é convertido para o formato numérico esperado pelo Keras: **0 para `Ativo` e 1 para `Inativo`**.

In [ ]:
df_modelo = df.drop(columns=['Numero_Cliente'])

colunas_categoricas = [
    'Genero',
    'Escolaridade',
    'Estado_Civil',
    'Faixa_Renda_Anual',
    'Categoria_Cartao'
]

# Transforma variáveis categóricas em variáveis binárias
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = encoder.fit_transform(df[colunas_categoricas])
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(colunas_categoricas))

# Mantém o dataframe original mas remove as colunas categóricas que foram codificadas
df_encoded = pd.concat([df.drop(columns=colunas_categoricas), encoded_df], axis=1)

X = df_encoded.drop('Target', axis=1)
y = (df_encoded['Target'] == 'Inativo').astype(int)

target_names = ['Ativo', 'Inativo']


Verificamos as dimensões finais dos dados após a codificação. O número de colunas de `X` agora é maior do que o original, pois cada variável categórica foi expandida em várias colunas binárias:

In [ ]:
print(f'Formato de X: {X.shape}')
print(f'Formato de y: {y.shape}')


### **Separando treino e teste**

Assim como nos modelos anteriores, dividimos os dados em treino (80%) e teste (20%), mantendo a proporção original das classes com `stratify=y`:

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)


### **Padronizando as variáveis numéricas**

Diferente das árvores, que são **insensíveis à escala** das variáveis, redes neurais são fortemente afetadas por variáveis em ordens de grandeza muito diferentes (por exemplo, idade em dezenas e valor de transação em milhares). Sem padronização, o gradiente descendente tem dificuldade em convergir e o treinamento se torna instável.

Aplicando a padronização. O `StandardScaler` aprende a escala usando apenas o treino e depois aplica a mesma transformação no teste:

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)


Inspecionando rapidamente o resultado da padronização — os valores agora têm média próxima de zero e desvio padrão próximo de um:

In [ ]:
X_treino_scaled


In [ ]:
X_teste_scaled


### **Lidando com o desbalanceamento das classes**

Como existem menos clientes `Inativo`, vamos calcular pesos simples para as classes. Isso ajuda a rede a não ignorar a classe minoritária durante o treinamento:

In [ ]:
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_treino
)

class_weight = {0: pesos[0], 1: pesos[1]}
class_weight


##**Montando uma rede neural simples**

A rede abaixo tem apenas uma camada intermediária pequena e uma saída para prever a probabilidade de churn. A saída será interpretada como:

* perto de 0: maior chance de cliente `Ativo`;
* perto de 1: maior chance de cliente `Inativo`.

In [ ]:
modelo_rede = keras.Sequential([
    keras.layers.Input(shape=(X_treino_scaled.shape[1],)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

modelo_rede.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'recall',
        keras.metrics.AUC(name='auc')
    ]
)

modelo_rede.summary()


##**Treinando a rede**



In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)

historico = modelo_rede.fit(
    X_treino_scaled,
    y_treino,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    class_weight=class_weight,
    verbose=1
)


Visualizando a evolução da rede ao longo do treinamento. A ideia é observar se o desempenho no treino e na validação caminham de forma parecida:

In [ ]:
historico_df = pd.DataFrame(historico.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

historico_df[['loss', 'val_loss']].plot(ax=axes[0])
axes[0].set_title('Loss durante o treinamento')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')

historico_df[['auc', 'val_auc']].plot(ax=axes[1])
axes[1].set_title('AUC durante o treinamento')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('AUC')

plt.tight_layout()
plt.show()


##**Avaliando no conjunto de teste**
Agora vamos avaliar a rede em dados que ela não viu durante o treinamento. Como a rede retorna probabilidades, usamos o ponto de corte de 0,5 para transformar a probabilidade em classe prevista:

In [ ]:
probabilidades_teste = modelo_rede.predict(X_teste_scaled).ravel()
predicoes_teste = (probabilidades_teste >= 0.5).astype(int)

acuracia_teste = accuracy_score(y_teste, predicoes_teste)
roc_auc_teste = roc_auc_score(y_teste, probabilidades_teste)

print(f'Acurácia em teste: {acuracia_teste:.3f}')
print(f'ROC-AUC em teste:  {roc_auc_teste:.3f}')
print()
print('Relatório de classificação:')
print(classification_report(
    y_teste,
    predicoes_teste,
    target_names=target_names,
    digits=3,
    zero_division=0
))


Matriz de confusão normalizada para observar os acertos e erros em cada classe:

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_teste,
    predicoes_teste,
    display_labels=target_names,
    normalize='true',
    cmap='Blues',
    values_format='.2f'
)
plt.title('Matriz de confusão - Rede Neural simples')
plt.show()


Curva ROC para avaliar a capacidade da rede de separar clientes ativos e inativos:

In [ ]:
RocCurveDisplay.from_predictions(
    y_teste,
    probabilidades_teste,
    name='Rede Neural simples'
)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Curva ROC - Rede Neural simples')
plt.show()
